# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un model de inteligență artificială antrenat pe cantități uriașe de text, capabil să înțeleagă, să genereze și să manipuleze limbajul uman. Aceste modele pot realiza sarcini variate precum traducerea, rezumarea și răspunsul la întrebări cu o fluență impresionantă.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități uriașe de text și cod, capabilă să înțeleagă, să genereze și să manipuleze limbajul uman într-un mod coerent și contextual. Aceste modele folosesc arhitecturi complexe, precum cele bazate pe transformere, pentru a identifica modele, relații și nuanțe în datele lingvistice, permițându-le să îndeplinească o gamă largă de sarcini, de la răspunsuri la întrebări și traduceri, la scrierea creativă și generarea de cod.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [7]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani, concentrandu-te pe activitatea politicienilor impotriva sistemului.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica românească a fost marcată de încercări ale unor politicieni de a modifica legislația penală și de a limita independența justiției. Aceste acțiuni au generat proteste stradale și critici din partea instituțiilor europene, subliniind tensiunile dintre puterea politică și statul de drept.

--- Gemini 2.5 Flash ---
În ultimii cinci ani, scena politică românească a fost marcată de ascensiunea unor noi partide, precum USR și AUR, care s-au poziționat ca alternative la formațiunile tradiționale, promovând discursuri anti-sistem și anti-corupție. Aceste forțe au obținut reprezentare parlamentară semnificativă, influențând dezbaterile publice și generând o polarizare crescută, în contextul unor reforme legislative și al luptei împotriva corupției.

--- OpenRouter Free ---
În ultimii 5 ani, politicieni auto-poziționați anti-sistem au depus repetat moțiuni de cenzură și au contestat deciziile coaliției PSD-PNL.
Partidele lor, AUR și SOS Ro

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [8]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii sunt dezamagitori, iar oamenii simpli plătesc pierd increderea in sistem. Nimeni nu mai ascultă poporul, politicienii au interese ascunse."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic
Emoție dominantă: Dezamăgire, neîncredere
Țintă principală: Clasa politică
Populism: da

--- Gemini 2.5 Flash ---
Ton: Critic, acuzator, deziluzionat.
Emoție dominantă: Frustrare, neîncredere.
Țintă principală: Politicienii, sistemul politic.
Populism: da

--- OpenRouter Free ---
Ton: Negativ  
Emoție dominantă: Răbdare  
Țintă principală: Critică față de corupția politică și distrugerea fidei în instituții  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [10]:
COMENTARIU = "Toți politicienii sunt dezamagitori, iar oamenii simpli plătesc pierd increderea in sistem. Nimeni nu mai ascultă poporul, politicienii au interese ascunse."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o dezamăgire profundă față de politicieni, sugerând că aceștia ignoră poporul și au interese ascunse, ceea ce duce la pierderea încrederii în sistem.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă dezamăgire față de toți politicienii, acuzându-i de interese ascunse și de ignorarea poporului, ceea ce duce la pierderea încrederii în sistem. Folosește un limbaj populist prin contrastul dintre 'oamenii simpli' și 'politicieni'."}

--- OpenRouter Free ---
{'emotie_dominanta': 'dezamagire', 'explicatie_scurta': 'Comentariul exprima o profunda frustrare si lipsa de incredere in clasa politica, perceputa ca fiind detasata de nevoile cetatenilor si ghidata exclusiv de interese personale.', '

#
Aici openrouter e limitat

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [12]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru sistemul politic.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Acest proces poate influența stabilitatea guvernamentală și procesul decizional pe termen scurt.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate duce la necesitatea organizării de noi scrutinuri, ceea ce implică o perioadă de incertitudine politică și resurse suplimentare. Această decizie poate influența dinamica partidelor politice și poate genera reconfigurări ale alianțelor politice în contextul noilor competiții electorale.

temperature=1.2:
Anularea alegerilor de către Curtea Constituțională poate duce la necesitatea organizării de noi alegeri, ceea ce poate genera un proces electoral prelungit și incertitudine politică. Acest lucru poate influența formar

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:** GEMINI 2.5 FLASH  
**Model de rezervă:**  GEMINI 2.5 FLASH LITE
**Temperature recomandată:** 0.7
**De ce am ales acest model?** Raspunsurile sunt corecte gramatical, exhaustive intr-un mod scurt in acelasi timp, acoperind tema in mod adecvat. Adnotarile sunt corecte si complete. Temperatura aleasa e pentru economisire de tokens
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [13]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales